> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.

# Chapter 5 — Deep Agents: Planning, Subagents & Filesystem Context (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2005.%20Building%20Personal%20Assistants/LC4LSH_Chapter_5_DeepAgents_Scientific_Overview.ipynb)

**Learning objectives**
- Explain the deep-agent pattern (planning + subagents + virtual filesystem)
- Decompose a research goal into subagent tasks
- Use a scratch filesystem for intermediate context
- Coordinate a supervisor over subagents

> Runtime: ~12 min (API)  
> Cost: paid LLM required  
> Data: synthetic research goal


## Environment setup


### Secrets (Colab or local)


In [ ]:
import os
try:
    from google.colab import userdata  # type: ignore
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False
if not IN_COLAB:
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass

def get_secret(name, default=None):
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)

API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"
if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret("LC4LSH_ANTHROPIC_API_KEY", "sk-ant-...")
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")
print("API keys loaded for", API_KEY_PROVIDER)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""


### Install pinned dependencies


In [ ]:
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "langchain-community==0.4.0" "langgraph>=0.2" "pydantic>=2.5" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)


In [ ]:
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "lsv2_pt_...")
LANGSMITH_PROJECT = "lc4lsh-chapter5-deepagents"
REGION = "US"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith("lsv2_"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF")


## What are "deep agents"?

**Deep agents** extend a simple tool-calling agent with three capabilities:

1. **Planning** — write and maintain an explicit plan (todo list)
2. **Subagents** — spawn focused sub-agents with their own context for subtasks
3. **Filesystem context** — a scratch workspace (real or virtual) to store intermediate results too large for the prompt

This pattern (popularized by `deepagents`) helps agents tackle long-horizon scientific tasks without overflowing context.


## 1. A virtual filesystem for context


In [ ]:
class VFS:
    """A tiny in-memory filesystem the agent can read/write."""
    def __init__(self):
        self.files = {}
    def write(self, path, content):
        self.files[path] = content
    def read(self, path):
        return self.files.get(path, "")
    def ls(self):
        return list(self.files)

vfs = VFS()
vfs.write("/notes/background.md", "# Background
Kinase Y is implicated in pathway Z.")
print("VFS files:", vfs.ls())


## 2. A planner that writes a todo plan


In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def make_plan(goal):
    prompt = (f"Goal: {goal}
Break it into 3-4 concrete subtasks as a numbered list. "
              f"Each subtask should be self-contained for a subagent.")
    plan = llm.invoke(prompt).content
    vfs.write("/plan.md", plan)
    return plan

plan = make_plan("Assess whether Compound X is a viable kinase-Y inhibitor")
print(plan)


## 3. Subagents with isolated context


In [ ]:
def subagent(name, task, context_paths=()):
    """Run a focused subagent; give it only the files it needs."""
    ctx = "

".join(vfs.read(p) for p in context_paths if vfs.read(p))
    prompt = f"You are subagent '{name}'.
Context:
{ctx}

Task: {task}
Produce a concise result."
    out = llm.invoke(prompt).content
    vfs.write(f"/results/{name}.md", out)
    return out

r1 = subagent("literature", "Summarize known kinase-Y inhibitors", ["/notes/background.md"])
r2 = subagent("docking", "Estimate binding feasibility of Compound X", [])
print("LITERATURE:
", r1[:300])
print("
DOCKING:
", r2[:300])


## 4. Supervisor synthesizes subagent results


In [ ]:
def supervisor(goal):
    results = "

".join(f"### {p}
{vfs.read(p)}" for p in vfs.ls() if p.startswith("/results/"))
    prompt = (f"Goal: {goal}

Subagent results:
{results}

"
              f"Synthesize a final verdict with confidence and caveats.")
    verdict = llm.invoke(prompt).content
    vfs.write("/final.md", verdict)
    return verdict

verdict = supervisor("Assess whether Compound X is a viable kinase-Y inhibitor")
print(verdict)
print("
VFS now:", vfs.ls())


## How this maps to `deepagents`

- **Planning** → the `/plan.md` todo written by `make_plan`
- **Subagents** → `subagent(...)` calls with isolated context windows
- **Filesystem** → the `VFS` scratch space decoupling intermediate results from the prompt

In the real `deepagents` library these are provided as middleware over a LangGraph agent, with a built-in `ls`/`read`/`write`/`edit` toolset and a todo-tracking tool.


## Limitations & safety notes

- This is a didactic re-implementation, not the `deepagents` package; install it for production features (middleware, real todo tools).
- Subagent isolation here is manual (we pass only selected files).
- The VFS is in-memory; use a real workspace dir for persistence.
- **Paid API required**.


In [ ]:
# Cleanup
import gc
for _v in ("llm", "model", "agent", "graph", "app", "workflow"):
    globals().pop(_v, None)
gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Why use a filesystem for agent context?</summary>It offloads large intermediate results from the prompt, preventing context-window overflow on long tasks.</details>

<details><summary>Why isolate subagent context?</summary>Each subagent sees only what it needs, reducing cost and distraction and avoiding cross-talk.</details>

<details><summary>What does the supervisor do?</summary>It aggregates subagent outputs into a coherent final answer and maintains the overall plan.</details>

### Tasks
- **Task A** - Add an `edit(path, old, new)` VFS method and use it to refine `/plan.md`.
- **Task B** - Let the supervisor spawn a third subagent dynamically if a result is inconclusive.
- **Task C** - Persist the VFS to a real temp directory and reload it.
- **Task D** - Install `deepagents` and re-implement this workflow using its `create_deep_agent`.
